In [ ]:
import os
from pathlib import Path

activity_path = Path.home() / "Documents" / "Activity_5_Files"
files = [f for f in activity_path.glob("*.txt")]
stats = []

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]
        text = " ".join(lines)
        words = text.split()
        
        line_count = len(lines)
        word_count = len(words)
        char_count = len(text)
        
        # Calculations
        wpl = word_count / line_count if line_count > 0 else 0
        cpw = char_count / word_count if word_count > 0 else 0
        
        stats.append({
            "name": file_path.name,
            "wpl": wpl,
            "cpw": cpw,
            "density": word_count / char_count if char_count > 0 else 0
        })

# Sort by word density (highest to lowest)
stats.sort(key=lambda x: x['density'], reverse=True)

print(f"{'Filename':<25} | {'Words/Line':<10} | {'Chars/Word':<10}")
print("-" * 50)
for s in stats:
    print(f"{s['name']:<25} | {s['wpl']:<10.2f} | {s['cpw']:<10.2f}")


In [ ]:
import csv
import string
from pathlib import Path
from collections import Counter

# --- CONFIGURATION ---
student_id = "2025-0355"
activity_path = Path.home() / "Documents" / "Activity_5_Files"
stopwords_file = activity_path / "stopwords.txt"
output_csv = activity_path / f"word_frequency_{student_id}.csv"

# 1. Create a default stopwords list if the file is missing
default_stopwords = {"the", "is", "and", "to", "in", "it", "of", "for", "with", "on", "a", "an", "this"}
if stopwords_file.exists():
    user_stopwords = set(stopwords_file.read_text(encoding="utf-8").lower().split())
    stopwords = default_stopwords.union(user_stopwords)
else:
    stopwords = default_stopwords

# --- PROCESSING ---
overall_counts = Counter()
per_file_counts = {}

files = list(activity_path.glob("*.txt"))

for file_path in files:
    try:
        # Normalize text: lowercase and remove punctuation
        text = file_path.read_text(encoding="utf-8").lower()
        # Create a translation table to remove punctuation
        text = text.translate(str.maketrans("", "", string.punctuation))
        
        # Filter stopwords
        words = [word for word in text.split() if word not in stopwords]
        
        # Count frequencies
        file_freq = Counter(words)
        per_file_counts[file_path.name] = file_freq
        overall_counts.update(file_freq)
        
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")

# --- OUTPUT ---
print(f"\n{'--- TOP MEANINGFUL WORDS (OVERALL) ---':^50}")
print(f"{'Word':<20} | {'Frequency':<10}")
print("-" * 35)

# Display Top 10 most frequent words
for word, count in overall_counts.most_common(10):
    print(f"{word:<20} | {count:<10}")

# --- EXPORT TO CSV ---
try:
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Word", "Frequency"])
        for word, count in overall_counts.most_common():
            writer.writerow([word, count])
    print(f"\nFull frequency data exported to: {output_csv.name}")
except Exception as e:
    print(f"CSV Export failed: {e}")

In [ ]:
import shutil
import os
from datetime import datetime
from pathlib import Path

# --- CONFIGURATION ---
student_id = "2025-0355"
student_name = "John Paulo R. Malabrigo"

# Define Paths
base_dir = Path.home() / "Documents" / "Activity_5_Files"
backup_dir = base_dir / f"backup_{student_id}"
log_file = backup_dir / f"backup_log_{student_id}.txt"

# Ensure directories exist
backup_dir.mkdir(parents=True, exist_ok=True)

def smart_backup(target_filename):
    target_path = base_dir / target_filename
    
    # 1. Check if the original file exists
    if not target_path.exists():
        print(f"Error: Original file '{target_filename}' not found in {base_dir}")
        return

    # 2. Generate Timestamp and Unique Backup Name
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_filename = f"{target_path.stem}_{student_id}_{timestamp}{target_path.suffix}"
    backup_path = backup_dir / backup_filename

    try:
        # 3. Perform the Copy
        shutil.copy2(target_path, backup_path)
        file_size = backup_path.stat().st_size

        # 4. Update the Personalized Log File
        log_exists = log_file.exists()
        with open(log_file, "a", encoding="utf-8") as log:
            if not log_exists:
                log.write(f"--- BACKUP HISTORY FOR {student_name} ({student_id}) ---\n")
            
            log.write("-" * 50 + "\n")
            log.write(f"Timestamp:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            log.write(f"Original File:  {target_filename}\n")
            log.write(f"Size:           {file_size} bytes\n")
            log.write(f"Backup Path:    {backup_path}\n")

        # 5. Output Confirmation
        print(f"Backup completed for {student_id} ({student_name})")
        print(f"File saved as: {backup_filename}")
        print(f"Log updated at: {log_file}")

    except Exception as e:
        print(f"An error occurred during backup: {e}")

# --- EXECUTION ---
# Change this filename to test different files in your folder
smart_backup(f"intro_{student_id}.txt")